In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

from langchain.chat_models import init_chat_model

model = init_chat_model("openai/gpt-oss-20b", model_provider="groq")

d:\agenticai\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import SystemMessage, HumanMessage

agent = create_agent(
    model=model,
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model=model,
            trigger=("messages", 10),
            keep=("messages",4)
        )
    ]
)

In [4]:
config={"configurable":{"thread_id":"test-1"}}

In [5]:
questions = [
    "What is 2+2?",
    "What is 10*5?",
    "What is 100/4?",
    "What is 15-7?",
    "What is 3*3?",
    "What is 4*4?",
]

for q in questions:
    response=agent.invoke({"messages":[HumanMessage(content=q)]},config)
    print(f"Messages: {response}")
    print(f"Messages: {len(response['messages'])}")

Messages: {'messages': [HumanMessage(content='What is 2+2?', additional_kwargs={}, response_metadata={}, id='016812a5-a49d-419d-bf3f-43d17564e8a9'), AIMessage(content='2\u202f+\u202f2\u202f=\u202f4.', additional_kwargs={'reasoning_content': 'User asks: "What is 2+2?" Simple arithmetic: 4. Provide answer.'}, response_metadata={'token_usage': {'completion_tokens': 40, 'prompt_tokens': 78, 'total_tokens': 118, 'completion_time': 0.041358959, 'completion_tokens_details': {'reasoning_tokens': 21}, 'prompt_time': 0.00377784, 'prompt_tokens_details': None, 'queue_time': 0.052862005, 'total_time': 0.045136799}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_c5a89987dc', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019e66a1-f1f7-7651-bbbe-236f22915988-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 78, 'output_tokens': 40, 'total_tokens': 118, 'output_token_details': {'reasoning': 21}})]}


In [8]:
from langchain_core.tools import tool

@tool
def search_hotels(city: str) -> str:
    """Search hotels - returns long response to use more tokens."""
    return f"""Hotels in {city}:
    1. Grand Hotel - 5 star, $350/night, spa, pool, gym
    2. City Inn - 4 star, $180/night, business center
    3. Budget Stay - 3 star, $75/night, free wifi"""

agent = create_agent(
    model=model,
    tools=[search_hotels],
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model=model,
            trigger=("tokens",600),
            keep=("tokens",250)
        )
    ]
)

In [11]:
config1 = {"configurable" : {"thread-id":"test-2"}}

def count_tokens(messages):
    total_chars = sum(len(str(m.content)) for m in messages)
    return total_chars // 4

In [12]:
cities = ["Paris", "London", "Tokyo", "New York", "Dubai", "Singapore"]

for city in cities:
    response = agent.invoke(
        {"messages": [HumanMessage(content=f"Find hotels in {city}")]},
        config=config
    )
    
    tokens = count_tokens(response["messages"])
    print(f"{city}: ~{tokens} tokens, {len(response['messages'])} messages")
    print(f"{(response['messages'])}")

Paris: ~334 tokens, 6 messages
[HumanMessage(content='Find hotels in Paris', additional_kwargs={}, response_metadata={}, id='e7c73abb-b605-44c3-a940-b8a871dc153a'), AIMessage(content='', additional_kwargs={'reasoning_content': 'The user wants to find hotels in Paris. We have a function to search hotels, requiring a city argument. We\'ll call the function with city: "Paris". Then return the result. The function is called via the tool.', 'tool_calls': [{'id': 'fc_286a6055-f5a1-42bb-aa36-ed76a1af6902', 'function': {'arguments': '{"city":"Paris"}', 'name': 'search_hotels'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 70, 'prompt_tokens': 129, 'total_tokens': 199, 'completion_time': 0.072234925, 'completion_tokens_details': {'reasoning_tokens': 46}, 'prompt_time': 0.006331208, 'prompt_tokens_details': None, 'queue_time': 0.054277317, 'total_time': 0.078566133}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_5c8ca06ea1', 'service_tier': 'on_d

In [14]:
@tool
def search_hotels(city: str) -> str:
    """Search hotels."""
    return f"Hotels in {city}: Grand Hotel $350, City Inn $180, Budget Stay $75"

agent = create_agent(
    model=model,
    tools=[search_hotels],
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model=model,
            trigger=("fraction", 0.005),
            keep=("fraction", 0.002),    
        ),
    ],
)

config = {"configurable": {"thread_id": "test-1"}}

def count_tokens(messages):
    return sum(len(str(m.content)) for m in messages) // 4

cities = ["Paris", "London", "Tokyo", "New York", "Dubai", "Singapore"]

for city in cities:
    response = agent.invoke(
        {"messages": [HumanMessage(content=f"Hotels in {city}")]},
        config=config
    )
    tokens = count_tokens(response["messages"])
    fraction = tokens / 128000  
    print(f"{city}: ~{tokens} tokens ({fraction:.4%}), {len(response['messages'])} msgs")
    print(response['messages'])

Paris: ~104 tokens (0.0813%), 4 msgs
[HumanMessage(content='Hotels in Paris', additional_kwargs={}, response_metadata={}, id='12cfb47e-95d8-4a83-b31d-614c8fede224'), AIMessage(content='', additional_kwargs={'reasoning_content': 'The user: "Hotels in Paris". We need to respond. The instruction: we have a tool to search hotels. We should use the function search_hotels with city="Paris". Then respond with the results. The system says "When you need to use a tool, respond with the tool invocation only." So we must call the function.', 'tool_calls': [{'id': 'fc_be872185-0caf-49fc-bc3e-9b8de5f2e686', 'function': {'arguments': '{"city":"Paris"}', 'name': 'search_hotels'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 94, 'prompt_tokens': 120, 'total_tokens': 214, 'completion_time': 0.096368859, 'completion_tokens_details': {'reasoning_tokens': 70}, 'prompt_time': 0.005856234, 'prompt_tokens_details': None, 'queue_time': 0.053809271, 'total_time': 0.102225093},

In [15]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver

def read_email_tool(email_id: str) -> str:
    """Mock function to read an email by its ID."""
    return f"Email content for ID: {email_id}"

def send_email_tool(recipient: str, subject: str, body: str) -> str:
    """Mock function to send an email."""
    return f"Email sent to {recipient} with subject '{subject}'"

In [16]:
agent=create_agent(
    model=model,
    tools=[read_email_tool,send_email_tool],
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email_tool":{
                    "allowed_decisions":["approve","edit","reject"]
                },
                "read_email_tool":False,

            }
        )
    ]
)

In [17]:
config = {"configurable": {"thread_id": "test-approve"}}

result = agent.invoke(
    {"messages": [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you?'")]},
    config=config
)

In [18]:
result

{'messages': [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you?'", additional_kwargs={}, response_metadata={}, id='48e9aaa0-9f25-4ed2-878d-e3d659a4bf7e'),
  AIMessage(content='', additional_kwargs={'reasoning_content': 'We need to use send_email_tool.', 'tool_calls': [{'id': 'fc_9563edf9-2160-4385-96c7-3f7a559bbf6d', 'function': {'arguments': '{"body":"How are you?","recipient":"john@test.com","subject":"Hello"}', 'name': 'send_email_tool'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 46, 'prompt_tokens': 174, 'total_tokens': 220, 'completion_time': 0.0476317, 'completion_tokens_details': {'reasoning_tokens': 9}, 'prompt_time': 0.00988915, 'prompt_tokens_details': None, 'queue_time': 0.052096724, 'total_time': 0.05752085}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_80501ff3a1', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_r

In [19]:
from langgraph.types import Command

if "__interrupt__" in result:
    print("⏸️ Paused! Approving...")
    
    result = agent.invoke(
        Command(
            resume={
                "decisions": [
                    {"type": "approve"}
                ]
            }
        ),
        config=config
    )
    
    print(f"✅ Result: {result['messages'][-1].content}")

⏸️ Paused! Approving...
✅ Result: ✅ Email sent to john@test.com with subject “Hello” and body “How are you?”


In [20]:
result

{'messages': [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you?'", additional_kwargs={}, response_metadata={}, id='48e9aaa0-9f25-4ed2-878d-e3d659a4bf7e'),
  AIMessage(content='', additional_kwargs={'reasoning_content': 'We need to use send_email_tool.', 'tool_calls': [{'id': 'fc_9563edf9-2160-4385-96c7-3f7a559bbf6d', 'function': {'arguments': '{"body":"How are you?","recipient":"john@test.com","subject":"Hello"}', 'name': 'send_email_tool'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 46, 'prompt_tokens': 174, 'total_tokens': 220, 'completion_time': 0.0476317, 'completion_tokens_details': {'reasoning_tokens': 9}, 'prompt_time': 0.00988915, 'prompt_tokens_details': None, 'queue_time': 0.052096724, 'total_time': 0.05752085}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_80501ff3a1', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_r

In [29]:
agent=create_agent(
    model=model,
    tools=[read_email_tool,send_email_tool],
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email_tool":{
                    "allowed_decisions":["approve","edit","reject"]
                },
                "read_email_tool":False,

            }
        )
    ]
)

In [30]:
config = {"configurable": {"thread_id": "test-reject"}}

result = agent.invoke(
    {"messages": [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you?'")]},
    config=config)

In [31]:
if "__interrupt__" in result:
    print("⏸️ Paused! Approving...")
    
    result = agent.invoke(
        Command(
            resume={
                "decisions": [
                    {"type": "reject"}
                ]
            }
        ),
        config=config
    )
    
    print(f"✅ Result: {result['messages'][-1].content}")

⏸️ Paused! Approving...
✅ Result: I’m sorry, but I’m not able to send the email right now. If you’d like me to try again or if there’s another way I can help, just let me know!


In [32]:
result


{'messages': [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you?'", additional_kwargs={}, response_metadata={}, id='b63c0cb9-3c36-477f-b47c-162110cd3c42'),
  AIMessage(content='', additional_kwargs={'reasoning_content': 'The user wants to send an email. We have a tool to send email: send_email_tool. We must call it. Use the function.', 'tool_calls': [{'id': 'fc_f0db2295-9062-4456-a8db-e6dbe207b5f7', 'function': {'arguments': '{"body":"How are you?","recipient":"john@test.com","subject":"Hello"}', 'name': 'send_email_tool'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 67, 'prompt_tokens': 174, 'total_tokens': 241, 'completion_time': 0.067929562, 'completion_tokens_details': {'reasoning_tokens': 30}, 'prompt_time': 0.008367655, 'prompt_tokens_details': None, 'queue_time': 0.047331905, 'total_time': 0.076297217}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_80501ff3a1', 'service_tier': 'on_deman

In [2]:
model = init_chat_model("llama-3.1-8b-instant", model_provider="groq")

In [24]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver


def read_email_tool(email_id: str) -> str:
    """Mock function to read an email by its ID."""
    return f"Email content for ID: {email_id}"

def send_email_tool(recipient: str, subject: str, body: str) -> str:
    """Mock function to send an email."""
    return f"Email sent to {recipient} with subject '{subject}'"

agent = create_agent(
    model=model,
    tools=[read_email_tool,send_email_tool],
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email_tool": {
                    "allowed_decisions": ["approve", "edit", "reject"],
                },
                "read_email_tool": False,
            }
        ),
    ],
)

In [25]:
from langchain_core.messages import HumanMessage

config = {"configurable": {"thread_id": "test-edit"}}

result = agent.invoke(
    {"messages": [HumanMessage(content="Send email to wrong@email.com with subject 'Test' and body 'Hello'")]},
    config=config
)

In [26]:
result

{'messages': [HumanMessage(content="Send email to wrong@email.com with subject 'Test' and body 'Hello'", additional_kwargs={}, response_metadata={}, id='af23cb07-fe75-4840-b2be-9a9036245942'),
  AIMessage(content='', additional_kwargs={'tool_calls': [{'id': '2p57rpb53', 'function': {'arguments': '{"body":"Hello","recipient":"wrong@email.com","subject":"Test"}', 'name': 'send_email_tool'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 30, 'prompt_tokens': 307, 'total_tokens': 337, 'completion_time': 0.028195805, 'completion_tokens_details': None, 'prompt_time': 0.024003249, 'prompt_tokens_details': None, 'queue_time': 0.051099081, 'total_time': 0.052199054}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019e66c6-0fc3-7353-bf32-4197e49053b5-0', tool_calls=[{'name': 'send_email_tool', 'args': {'body': 'Hello', 

In [27]:
from langgraph.types import Command

if "__interrupt__" in result:
    print("⏸️ Paused! Editing...")
    
    result = agent.invoke(
        Command(
            resume={
                "decisions": [
                    {
                        "type": "edit",
                        "edited_action": {
                            "name": "send_email_tool",    
                            "args": {                   
                                "recipient": "correct@email.com",
                                "subject": "Corrected Subject",
                                "body": "This was edited by human before sending"
                            }
                        }
                    }
                ]
            }
        ),
        config=config
    )
    
    print(f"✏️ Result: {result['messages'][-1].content}")

⏸️ Paused! Editing...
✏️ Result: 


In [28]:
result

{'messages': [HumanMessage(content="Send email to wrong@email.com with subject 'Test' and body 'Hello'", additional_kwargs={}, response_metadata={}, id='af23cb07-fe75-4840-b2be-9a9036245942'),
  AIMessage(content='', additional_kwargs={'tool_calls': [{'id': '2p57rpb53', 'function': {'arguments': '{"body":"Hello","recipient":"wrong@email.com","subject":"Test"}', 'name': 'send_email_tool'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 30, 'prompt_tokens': 307, 'total_tokens': 337, 'completion_time': 0.028195805, 'completion_tokens_details': None, 'prompt_time': 0.024003249, 'prompt_tokens_details': None, 'queue_time': 0.051099081, 'total_time': 0.052199054}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019e66c6-0fc3-7353-bf32-4197e49053b5-0', tool_calls=[{'type': 'tool_call', 'name': 'send_email_tool', 'args